In [1]:
# !pip install playwright
# !playwright install

# MyCareersFuture Automated Daily Job Scraper & ETL Pipeline

## Description
This notebook implements an automated, asynchronous ETL (Extract, Transform, Load) pipeline that targets **MyCareersFuture**, Singapore's government-backed job portal. It monitors the job market in real time by scanning for specific high-growth technology roles posted within a strict **48-hour window (Today & Yesterday)**. 

By leveraging browser automation to bypass dynamically loaded JavaScript components, the notebook extracts operational data points and structuralizes them into analytical datasets (CSV format) for down-stream metrics tracking or automated job application workflows.

---

## Key Architectural Features

* **Asynchronous Web Extraction:** Utilizes the `playwright` Async API to launch headless browser contexts, mimicking real-user interaction and bypassing rigid content blockages on Modern Single Page Applications (SPAs).
* **Smart Pagination Loop:** Dynamically pages through search queries sorted by `new_posting_date`. The extraction loops autonomously break when chronological thresholds are surpassed (encountering items older than "yesterday") or when pages return empty DOM trees.
* **Resilient Element Fallbacks:** Implements fuzzy CSS attribute selectors and string-splitting techniques to dynamically extract company profiles and job payloads, neutralizing site-wide structural updates.
* **Pandas Data Transformation (ETL):** Aggregates cross-functional listings into unified distributed DataFrames, strips out monetary syntax string bloat (such as commas and currency symbols), and splits combined salary bounds (`$X,XXX to $Y,YYY`) into individual numeric boundaries (`Min_Pay` and `Max_Pay`).

---

## Pipeline Workflow & Configuration

1. **Target Matrix:** Target arrays are currently initialized to audit: 
   `["Data Analyst"]`
2. **Target Extraction Point:** `https://www.mycareersfuture.gov.sg/search`
3. **Extracted Payload:** Job Title, Company Name, Posting Window Category, Split Pay Matrix (`Min_Pay`, `Max_Pay`), Application Target URLs, and Search Keyword Origin tracking.
4. **Output Destination:** A timestamped local raw file path compiled daily: `mycareersfuture_jobs_YYYYMMDD.csv`.

In [2]:
import asyncio
import csv
from datetime import datetime
import time
from playwright.async_api import async_playwright
import pandas as pd

In [3]:
targeted_job_title_list = ["Data Analyst", 'AI', 'LLM', 'Data Scientist']

In [4]:
async def scrape_mycareersfuture(targeted_job_title):
    jobs_data = []
    page_number = 0
    has_more_recent_jobs = True

    print("Launching browser (Async)...")
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        
        await page.set_extra_http_headers({
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
        })

        # Pagination Loop
        while has_more_recent_jobs:
            url = f"https://www.mycareersfuture.gov.sg/search?search={targeted_job_title}&sortBy=new_posting_date&page={page_number}"
            print(f"Navigating to Page {page_number}: {url}")
            
            try:
                await page.goto(url, wait_until="domcontentloaded")
                
                # Selector strategy
                try:
                    await page.wait_for_selector('div[class*="job-card"], div[id^="job-card"]', timeout=10000)
                except Exception:
                    await page.wait_for_selector('div[class*="card"]', timeout=5000)

                job_cards = await page.query_selector_all('div[class*="job-card"], div[id^="job-card"]')
                if not job_cards:
                    job_cards = await page.query_selector_all('.card')

                if not job_cards:
                    print("No more job cards found on this page. Stopping.")
                    break

                recent_jobs_on_this_page = 0

                for card in job_cards:
                    try:
                        card_text = await card.inner_text()
                        card_text_lines = [line.strip() for line in card_text.split('\n') if line.strip()]
                        card_text_lower = card_text.lower()

                        is_recent = "posted today" in card_text_lower or "posted yesterday" in card_text_lower

                        if is_recent:
                            recent_jobs_on_this_page += 1
                            
                            # 1. Extract Job Title
                            title_element = await card.query_selector('[data-testid="job-card__job-title"], h3, h2')
                            title = await title_element.inner_text() if title_element else "N/A"
                            title = title.strip().split('\n')[0]

                            # 2. Extract Company Name
                            company = "N/A"
                            if card_text_lines:
                                company = card_text_lines[0]
                                if company.lower() == title.lower() and len(card_text_lines) > 1:
                                    company = card_text_lines[1]
                                if "posted" in company.lower() or "application" in company.lower():
                                    company = "N/A"

                            # 3. Get Date Text
                            date_text = "Today" if "today" in card_text_lower else "Yesterday"

                            # 4. Extract Pay Range / Salary (ADDED SECTIONS)
                            salary = "N/A"
                            salary_element = await card.query_selector('[class*="salary"], [id*="salary"]')
                            if salary_element:
                                salary = await salary_element.inner_text()
                                salary = salary.replace('\n', ' ').strip()
                            else:
                                # Fallback strategy: look for the line containing '$' inside text lines
                                for line in card_text_lines:
                                    if "$" in line:
                                        salary = line.strip()
                                        break

                            # 5. Extract Job URL
                            link_element = await card.query_selector('a')
                            href = await link_element.get_attribute("href") if link_element else ""
                            job_url = "https://www.mycareersfuture.gov.sg" + href if href.startswith('/') else href

                            jobs_data.append({
                                "Title": title,
                                "Company": company,
                                "Posted": date_text,
                                "Salary": salary,  # Included in payload
                                "URL": job_url,
                                'Search_job_title' : targeted_job_title.replace("%20", " ")
                            })
                    except Exception:
                        continue

                print(f"-> Page {page_number}: Found {recent_jobs_on_this_page} recent jobs out of {len(job_cards)} postings.")

                if recent_jobs_on_this_page == 0:
                    print("Reached jobs older than 'yesterday'. Stopping pagination.")
                    has_more_recent_jobs = False
                else:
                    page_number += 1
                    time.sleep(1.5)

            except Exception as e:
                print(f"Error processing page {page_number}: {e}")
                break

        await browser.close()

    if jobs_data:
        return jobs_data
    else:
        print("\nNo jobs matched 'today' or 'yesterday' across any pages.")
        return []


In [5]:
jobs_data_list = []

for c, i in enumerate(targeted_job_title_list):
    i = i.replace(" ", "%20")
    globals()[f"{i}_jobs_data"] = await scrape_mycareersfuture(i)

    jobs_data_list.append(globals()[f"{i}_jobs_data"])

for c,i in enumerate(jobs_data_list):
    if c == 0:
        df_jobs_data = pd.DataFrame(i)
    else:
        df_jobs_data = pd.concat([df_jobs_data, 
                                 pd.DataFrame(i)])

salary_split = df_jobs_data['Salary'].str.replace(r'\s+', '', regex=True).str.split('to', expand=True)
salary_split = df_jobs_data['Salary'].str.replace(r'$', '', regex=False).str.split('to', expand=True)

df_jobs_data['Min_Pay'] = salary_split[0].str.replace(',', '').str.strip()
df_jobs_data['Max_Pay'] = salary_split[1].str.replace(',', '').str.strip()

df_jobs_data = df_jobs_data.drop(columns=['Salary'])
df_jobs_data.to_csv(f'mycareersfuture_jobs_{datetime.now().strftime('%Y%m%d')}.csv',
                    index = 0)

Launching browser (Async)...
Navigating to Page 0: https://www.mycareersfuture.gov.sg/search?search=Data%20Analyst&sortBy=new_posting_date&page=0
-> Page 0: Found 9 recent jobs out of 40 postings.
Navigating to Page 1: https://www.mycareersfuture.gov.sg/search?search=Data%20Analyst&sortBy=new_posting_date&page=1
-> Page 1: Found 0 recent jobs out of 40 postings.
Reached jobs older than 'yesterday'. Stopping pagination.
Launching browser (Async)...
Navigating to Page 0: https://www.mycareersfuture.gov.sg/search?search=AI&sortBy=new_posting_date&page=0
-> Page 0: Found 20 recent jobs out of 40 postings.
Navigating to Page 1: https://www.mycareersfuture.gov.sg/search?search=AI&sortBy=new_posting_date&page=1
-> Page 1: Found 20 recent jobs out of 40 postings.
Navigating to Page 2: https://www.mycareersfuture.gov.sg/search?search=AI&sortBy=new_posting_date&page=2
-> Page 2: Found 20 recent jobs out of 40 postings.
Navigating to Page 3: https://www.mycareersfuture.gov.sg/search?search=AI&sor